# Lab 08 — Authoring Schemas & CI Hooks (Pandera + Pydantic)
### Week 2 · Data Engineering for LLM Pipelines

Lab 07 was the **why** — drift is silent, so gate it. This is the **how**. You'll
author *practical* contracts: column types with **regex** and **range** rules,
**cross-column** invariants, whole-**DataFrame** checks, custom **pydantic validators**,
and a real **pytest** hook so bad data trips a red build before it ever reaches an LLM
step.

Two tools, two seams — the same rules expressed each way:
- **pandera** — bulk **DataFrame** validation in ETL/ELT.
- **pydantic** — one **record** at a time, at an API boundary or a queue consumer.

**By the end you will be able to:**
1. Write pandera **column** checks (regex allow-lists, ranges, categorical sets) plus
   **DataFrame-level** and **cross-column** checks.
2. Write a **pydantic** row model with a **`@field_validator`** for custom cross-field logic.
3. Turn a `SchemaErrors` dump into a compact **roll-up** for CI and a **CSV** for triage.
4. Wire a **pytest** smoke test that fails fast on drift.

> **Hints stay light (Labs 05–07).** Each Part opens with a **Toolbox**; you assemble
> the pieces. Target **20/20**; a red check never halts the notebook. The pandera
> **check API moved** since most tutorials were written — the Toolbox lists the current
> spelling; watch for it.


## Setup — a feature table & the `check()` helper

In [ ]:
%pip install -r requirements.txt

In [ ]:

import numpy as np
import pandas as pd
import pandera.pandas as pa          # modern, warning-free import (NOT `import pandera as pa`)
from pydantic import BaseModel, Field, ValidationError, field_validator
from typing import Literal

ALLOWED  = ["USA", "DE", "SG", "BR"]
ID_RE    = r"^C\d{5}$"                          # customer id: C + 5 digits
EMAIL_RE = r"^[^@\s]+@[^@\s]+\.[^@\s]+$"        # pragmatic email shape

def build_cordwell(n=500, seed=1):
    """Clean Cordwell feature table (as if emitted by the Lab 06 pipeline)."""
    rng = np.random.default_rng(seed)
    age = rng.integers(16, 80, n).astype("int64")
    ltv = np.round(np.clip(rng.lognormal(3.0, 0.7, n), 0, 5e5), 2)
    return pd.DataFrame({
        "customer_id":   [f"C{i:05d}" for i in range(n)],
        "country_norm":  rng.choice(ALLOWED, n, p=[.55, .2, .15, .1]),
        "age":           age,
        "ltv_usd":       ltv,
        "email":         [f"user{i}@cordwell.example" for i in range(n)],
        "is_adult":      age >= 18,
        "is_high_value": ltv >= np.quantile(ltv, 0.90),
    })

users = build_cordwell()
print("shape:", users.shape, "| minors (age<18):", int((users["age"] < 18).sum()))
print(users.dtypes.to_string())
users.head(3)

In [ ]:

# ── soft self-check: prints PASS/FAIL, never raises ──────────────────────────
_score = {"pass": 0, "fail": 0}
def check(label, predicate):
    try:
        ok = bool(predicate() if callable(predicate) else predicate); note = ""
    except Exception as e:
        ok, note = False, f"  [error: {type(e).__name__}: {e}]"
    _score["pass" if ok else "fail"] += 1
    print(f"{'\u2705 PASS' if ok else '\u274c FAIL'} \u2014 {label}{note}")
def score():
    t = _score["pass"] + _score["fail"]
    print(f"\n{'='*46}\n  {_score['pass']}/{t} checks passing  ({_score['fail']} to go)\n{'='*46}")
def raises(fn, exc=Exception):
    try:
        fn(); return False
    except exc:
        return True

In [ ]:

check("Setup: table is 500 x 7", lambda: users.shape == (500, 7))
check("Setup: 15 minors present (so cross-column rules have teeth)",
      lambda: int((users["age"] < 18).sum()) == 15)


---
## Part A — Author a pandera `DataFrameSchema`

Build the contract in three passes: column rules, then whole-frame rules, then read the
failure report.

> **🧰 Toolbox for Part A** — `pa.DataFrameSchema({cols}, checks=[...])` ·
> `pa.Column(dtype, check, nullable=, unique=)` · `pa.Check.str_matches(regex)` ·
> `pa.Check.isin([...])` · `pa.Check.in_range(lo, hi)` · `pa.Check.ge(0)` · `pa.Int64` ·
> `pa.Check(lambda df: <bool>, error=...)` for whole-frame rules ·
> `schema.validate(df, lazy=True)` · `pa.errors.SchemaErrors` · `err.failure_cases`.
>
> ⚠️ **Two API traps.** (1) On pandas 3.0 string columns are **`str`**, not `object` —
> `pa.Column(object, …)` rejects clean data. (2) `DataFrameSchema` has **no**
> `update_checks` / `add_checks` methods — declare whole-frame rules in the `checks=[...]`
> argument at construction time.


### A1 — Column types & constraints

Write `UsersSchema` so every column carries its dtype **and** its rule: `customer_id`
matches `^C\d{5}$` and is **unique**; `country_norm` in the allowed set; `age` an integer
in `[0, 120]`; `ltv_usd` a float `≥ 0`; `email` matches `EMAIL_RE`; the two flags are
booleans. Nothing nullable.

In [ ]:

# TODO: build UsersSchema — one Column per field with the correct dtype AND its rule.
#       (str columns are `str` not `object`; put the whole-frame rules in Part A2.)
UsersSchema = pa.DataFrameSchema({
    "customer_id": pa.Column(str),
})
print("columns:", list(UsersSchema.columns))

In [ ]:

check("A1: schema covers all 7 columns",
      lambda: set(UsersSchema.columns) == {"customer_id","country_norm","age","ltv_usd","email","is_adult","is_high_value"})
check("A1: accepts the clean table (dtype trap avoided) \u2014 500 rows",
      lambda: len(UsersSchema.validate(users, lazy=True)) == 500)
check("A1: rejects a malformed customer_id",
      lambda: raises(lambda: UsersSchema.validate(users.assign(customer_id=users["customer_id"].mask(users.index==0, "BADID")), lazy=True),
                     pa.errors.SchemaErrors))


### A2 — Whole-frame & cross-column rules

Some rules span columns or the whole frame. Rebuild `UsersSchema` **keeping your A1
columns** and adding two frame-level checks via `checks=[...]`:
1. **Cross-column consistency** — wherever `age >= 18`, `is_adult` must be `True`.
2. **Sanity band** — the *median* `ltv_usd` must be `≤ 100_000`.

*Tip:* a frame-level `pa.Check` receives the whole DataFrame; return a **single bool**
for a clean, one-line failure (a returned Series broadcasts the failure across every
column — noisy).

In [ ]:

# TODO: rebuild UsersSchema with columns=UsersSchema.columns and TWO frame-level checks:
#   (1) age>=18 implies is_adult is True   (2) median ltv_usd <= 100_000
#   Return a single bool from each check lambda.
UsersSchema = pa.DataFrameSchema(columns=UsersSchema.columns, checks=[])
print("frame-level checks:", len(UsersSchema.checks))

xbad = users.copy()
xbad.loc[(xbad["age"] >= 18).idxmax(), "is_adult"] = False
xcol_caught = raises(lambda: UsersSchema.validate(xbad, lazy=True), pa.errors.SchemaErrors)
print("cross-column violation caught:", xcol_caught)

In [ ]:

check("A2: two frame-level checks attached", lambda: len(UsersSchema.checks) == 2)
check("A2: enriched schema still accepts clean data", lambda: len(UsersSchema.validate(users, lazy=True)) == 500)
check("A2: cross-column rule catches an inconsistent adult flag", lambda: xcol_caught is True)


### A3 — Friendly error reporting

Given a `broken` frame with several column-level problems, validate it lazily, capture
`failure_cases` as `report`, and collapse it to a `rollup` (one row per
`(column, check)`, worst-first) — the artifact you'd post to CI.

In [ ]:

broken = users.copy()
broken.loc[[0, 10], "age"] = [200, 130]
broken.loc[[1, 11, 21], "email"] = "not-an-email"
broken.loc[[2, 12], "country_norm"] = ["usa", "Germany"]
broken.loc[3, "customer_id"] = "BADID"

# TODO: validate `broken` lazily, capture err.failure_cases -> report, then build
#       `rollup` = one row per (column, check), sorted worst-first.
report = None
rollup = pd.DataFrame(columns=["column", "check", "failures"])
rollup

In [ ]:

check("A3: 8 total failure cases captured", lambda: report is not None and len(report) == 8)
check("A3: roll-up worst offender is email (3)",
      lambda: rollup.iloc[0]["column"] == "email" and int(rollup.iloc[0]["failures"]) == 3)


---
## Part B — Pydantic row contracts

Same rules, per record. Pydantic shines at boundaries where data is a dict, not a frame.

> **🧰 Toolbox for Part B** — `class X(BaseModel)` · `Field(pattern=, ge=, le=)` ·
> `Literal[...]` · `@field_validator("f") @classmethod def v(cls, value, info): ...` ·
> `info.data` (fields validated **before** this one) · `Model(**row_dict)` ·
> `ValidationError` · `df.iloc[i].to_dict()`.


### B1 — A row model with a custom validator

Write `CustomerRow` mirroring the schema (regex id, `Literal` country, ranged age,
non-negative ltv, regex email, two bools). Add a **`@field_validator("is_adult")`** that
enforces the same cross-column rule: if `age >= 18`, `is_adult` must be `True`.

*Field order matters:* `info.data` only holds fields validated **before** the current
one — so `age` must be declared above `is_adult`.

In [ ]:

# TODO: define CustomerRow(BaseModel) mirroring the schema, plus a @field_validator on
#       is_adult enforcing: age>=18 -> is_adult must be True.
class CustomerRow(BaseModel):
    customer_id: str
    # ... the other six fields, with constraints, and the validator

good = None
bad_fields = []
xfield_raised = False
print("xfield_raised:", xfield_raised)

In [ ]:

check("B1: clean row validates", lambda: good is not None and good.customer_id == "C00000")
check("B1: bad row rejected on both age and email", lambda: bad_fields == ["age", "email"])
check("B1: cross-field validator rejects an inconsistent adult flag", lambda: xfield_raised is True)


### B2 — Validate a batch

Write `validate_batch(df, model)` that runs the model over every row and returns a
DataFrame of `{idx, error}` for the failures (empty if all pass). Run it on a frame with
two planted bad rows.

In [ ]:

def validate_batch(df, model):
    # TODO: run `model` over each row; collect {idx, error} for failures; return a DataFrame.
    return pd.DataFrame(columns=["idx", "error"])

batch = users.copy()
batch.loc[5, "email"] = "bad"
batch.loc[7, "country_norm"] = "XX"
batch_errors = validate_batch(batch, CustomerRow)
print("rows rejected:", len(batch_errors))
batch_errors

In [ ]:

check("B2: batch validator returns a DataFrame", lambda: isinstance(batch_errors, pd.DataFrame))
check("B2: exactly the 2 planted bad rows are caught", lambda: len(batch_errors) == 2)


---
## Part C — Error handling & a CI hook

Package validation so a pipeline (and a CI run) fails fast with evidence.

> **🧰 Toolbox for Part C** — `Path(...).mkdir(parents=True, exist_ok=True)` ·
> `try/except pa.errors.SchemaErrors` · `DataFrame.to_csv(path, index=False)` ·
> `raise RuntimeError(msg)` · `subprocess.run([...])` · `pytest`.


### C1 — `validate_or_artifact` (return-or-raise)

Implement it: on success return the validated frame; on failure write
`<name>_failures.csv` and raise a concise **`RuntimeError`** (not `SystemExit` — that
would kill the whole process, not just fail the step).

In [ ]:

from pathlib import Path

def validate_or_artifact(df, schema, name, out_dir="artifacts/validation"):
    """Return the validated frame, or write a CSV + raise RuntimeError on failure."""
    # TODO: mkdir; validate lazily; on SchemaErrors write <name>_failures.csv and raise
    #       RuntimeError with a compact top-issues summary; else return the frame.
    return schema.validate(df, lazy=True)

clean_out = validate_or_artifact(users, UsersSchema, "users_clean")

try:
    validate_or_artifact(broken, UsersSchema, "users_broken"); gate_blocked = False
except RuntimeError:
    gate_blocked = True
except Exception:
    gate_blocked = False
report_written = (Path("artifacts/validation") / "users_broken_failures.csv").exists()
print("gate blocked:", gate_blocked, "| CSV written:", report_written)

In [ ]:

check("C1: gate passes the clean batch (500 rows)", lambda: len(clean_out) == 500)
check("C1: gate raises RuntimeError on the broken batch", lambda: gate_blocked is True)
check("C1: gate wrote a CSV triage report", lambda: report_written is True)


### C2 — A pytest smoke test

CI needs a test that goes red on drift. Write `test_cordwell_schema.py` to disk — a
self-contained test that regenerates a clean frame and asserts it validates — then run
`pytest -q` on it and confirm it passes.

*In a real repo you'd import the shared schema module instead of redefining it; inlined
here so the file stands alone.*

In [ ]:

import subprocess, sys, textwrap
from pathlib import Path

# TODO: write a self-contained pytest file `tests/test_cordwell_schema.py` that
#       regenerates a clean frame and asserts it validates against a schema, then run
#       `pytest -q` on it via subprocess and set ci_passed = (returncode == 0).
Path("tests").mkdir(exist_ok=True)
ci_passed = False
print("CI smoke test passed:", ci_passed)

In [ ]:

check("C2: pytest smoke test file exists", lambda: Path("tests/test_cordwell_schema.py").exists())
check("C2: pytest run is green on clean data", lambda: ci_passed is True)
score()


---
## Wrap-up — answer in this Markdown cell

1. **One column check, one frame check.** Name one you wrote at each level and why the
   rule belongs there rather than the other.
2. **The boundary.** Give a concrete point in the Lab 05–08 pipeline where you'd reach
   for **pydantic** over **pandera**, and one where it's the reverse.
3. **CI roll-up.** Paste your A3 `rollup` — the compact artifact a failing build would show.

**Key takeaways**
- **Column rules pinpoint rows; frame rules assert invariants.** Regex/range/isin at the
  column level; uniqueness, cross-column consistency, and sanity bands at the frame level.
- Frame-level `pa.Check`s should return **one bool** — a returned Series broadcasts the
  failure across every column and clutters the report.
- `DataFrameSchema` has **no `add_checks`/`update_checks`** — pass `checks=[...]` at
  construction (a currency fix over older tutorials).
- **pydantic `@field_validator`** sees earlier fields via `info.data` — order your fields
  so dependencies come first.
- A gate **returns-or-raises** (`RuntimeError`, never `SystemExit`), and a **pytest**
  smoke test turns the whole contract into a red/green CI signal.

> ⚠️ **CURRENCY FLAG — `EmailStr`.** Pydantic's `EmailStr` is the production-grade email
> type, but it needs the extra dependency `pip install "pydantic[email]"`
> (the `email-validator` package). This lab uses a regex `Field(pattern=…)` to stay
> self-contained; swap in `EmailStr` once that dependency is in your environment.
